In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm

from loka.data import load_breakthrough_states
from loka.models import get_breakthrough_cnn
from loka.geometrical_methods import herding_order, k_center_order
from loka.embeddings import with_counts, with_near_goal, with_counts_and_near_goal

import os
import pandas as pd

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
filename = "/home/axelbjarkar/RU/T-404-LOKA-code/data/o.txt"
X_all, Y_all = load_breakthrough_states(filename, two_channel=True)
total_data_amount = X_all.shape[0]
total_data_amount

In [ ]:
data_dir = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/embeddings"
files = [f for f in os.listdir(data_dir) if f.endswith(".txt")]

emd_methods = [with_counts, with_near_goal, with_counts_and_near_goal]
emb_by_name = {emb.__name__: emb for emb in emd_methods}

def emb_for(name):
    base = name.rsplit("_", 1)[0]  # drop the "_10p" suffix
    for emb_name, emb in emb_by_name.items():
        if base.endswith(emb_name):
            return emb
    raise ValueError(f"no embedding method matches {name}")

orders = []
for f in sorted(files):
    name = f.removesuffix(".txt")
    orders.append((name, os.path.join(data_dir, f), emb_for(name)))

orders

In [ ]:
start = 0.01
end = 0.1
steps = 50
batch_size = 512
eval_bs = 1024
epochs = 50

results_dir = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/results/mse_all"
os.makedirs(results_dir, exist_ok=True)  # create the folder if it doesn't exist yet

splits = np.linspace(start, end, num=steps, endpoint=True)
Y_all_t = torch.FloatTensor(Y_all).to(device)

for name, path, emb in orders:
    csv_file = f'{results_dir}/{name}.csv'
    if os.path.exists(csv_file):
        print(f'skipping {name}: {csv_file} already exists')
        continue

    mse_values = []
    prc_values = []
    pbar = tqdm(total=steps * epochs, desc=f'training {name}')

    # load the ordered subset and apply the embedding it was selected with
    X, Y = load_breakthrough_states(path)
    X = emb(X)

    # full dataset through the same embedding, for evaluation
    X_all_t = torch.FloatTensor(emb(X_all)).to(device)

    for m, ratio in enumerate(splits):
        n_train = int(total_data_amount * ratio)
        X_train, Y_train = X[:n_train], Y[:n_train]  # first `ratio` of the data, in order

        model = get_breakthrough_cnn(X_train).to(device)  # channels sized to the embedding

        # training setup
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.MSELoss()

        # convert training data to tensors
        X_train_t = torch.FloatTensor(X_train).to(device)
        Y_train_t = torch.FloatTensor(Y_train).to(device)

        # training loop
        for epoch in range(epochs):
            perm = torch.randperm(len(X_train_t), device=device)  # new shuffle each epoch

            for i in range(0, len(X_train_t), batch_size):
                idx = perm[i:i + batch_size]
                xb, yb = X_train_t[idx], Y_train_t[idx]

                optimizer.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                optimizer.step()
            pbar.update(1)

        pbar.set_description(f'calculating total MSE ({name})')

        # batched evaluation
        sq_err, n = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(X_all_t), eval_bs):
                xb, yb = X_all_t[i:i + eval_bs], Y_all_t[i:i + eval_bs]
                sq_err += F.mse_loss(model(xb), yb, reduction='sum').item()  # total squared error in this chunk
                n += yb.numel()  # count of values summed (numel = number of elements, all dims)
        mse = sq_err / n  # total / count = mean, same as nn.MSELoss()
        mse_values.append(mse)
        prc_values.append(ratio)

        # free GPU memory before the next model
        del model, optimizer, X_train_t, Y_train_t
        torch.cuda.empty_cache()

        pbar.set_description(f'training {name}')
        pbar.set_postfix(model=f'{m+1}/{steps}', ratio=f'{ratio:.2f}',
                         loss=f'{loss.item():.4f}', mse=f'{mse:.6f}')

    pbar.close()

    # one row per split: training ratio and MSE
    pd.DataFrame({'ratio': prc_values, 'mse': mse_values}).to_csv(csv_file, index=False)